# 06｜Choice证券主数据与交易日历落库验收

本Notebook验证Choice参考数据的下一阶段闭环：

1. 使用Choice `css`获取证券名称、上市日期和退市日期；
2. 使用Choice `tradedates`获取沪深交易日，并扩展为包含休市日的完整自然日历；
3. 将数据幂等写入SQLite的`security_master`和`trading_calendar`；
4. 连续执行两次，确认主键记录数不增长；
5. 检查代码、名称、交易所、日期覆盖、周末开市、沪深日历差异；
6. 用已落库Choice日线反查交易日历，确认行情日期都是开市日；
7. 导出Excel、JSON和数据地图证据。

默认只验证`000001.SZ、601988.SH、510300.SH`三只证券。第一次不要直接拉取全部A股；样本通过后，再把`MASTER_SCOPE`改成`all_a`。

官方接口依据：<https://quantapi.eastmoney.com/Upload/EMQuantAPI_Python.html>

安全边界：Notebook不会打印或导出Choice账号、密码、Token或`userInfo`内容。

## 1. 定位项目并检查当前Python

In [1]:
import os
import sys
from pathlib import Path

print("当前Python：", sys.executable)
print("Python版本：", sys.version.split()[0])
print("Conda环境：", os.getenv("CONDA_DEFAULT_ENV", "未检测到"))
print("Notebook当前目录：", Path.cwd().resolve())

# Notebook位于项目notebooks目录时无需修改。
# 如果单独存放，再填写实际根目录；下划线前不要加反斜杠。
# PROJECT_ROOT_OVERRIDE = r"D:\OneDrive\桌面\qianji_openbb_mini"
PROJECT_ROOT_OVERRIDE = ""


def find_project_root(start: Path) -> Path:
    if PROJECT_ROOT_OVERRIDE.strip():
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if (candidate / "src" / "qianji_data_mini").exists():
            return candidate
        raise FileNotFoundError(f"指定项目根目录不正确：{candidate}")
    for candidate in (start, *start.parents):
        if (
            (candidate / "src" / "qianji_data_mini").exists()
            and (candidate / "extensions" / "openbb_choice").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "没有找到qianji_openbb_mini项目。请把Notebook放进项目notebooks目录，"
        "或填写PROJECT_ROOT_OVERRIDE。"
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
print("项目根目录：", PROJECT_ROOT)
print(".env存在：", (PROJECT_ROOT / ".env").exists())

当前Python： d:\minicoda3\envs\dm311\python.exe
Python版本： 3.11.14
Conda环境： dm311
Notebook当前目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\notebooks
项目根目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini
.env存在： True


## 2. 加载配置并检查06号所需代码版本

In [2]:
from importlib.metadata import PackageNotFoundError, version

from dotenv import load_dotenv
from packaging.version import Version

ENV_PATH = PROJECT_ROOT / ".env"
if ENV_PATH.exists():
    load_dotenv(ENV_PATH, override=True)
else:
    print("提示：没有找到.env，将使用当前进程已有环境变量。")


def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "未安装"


installed_qianji = package_version("qianji-data-mini")
print("qianji-data-mini版本：", installed_qianji)

if installed_qianji == "未安装" or Version(installed_qianji) < Version("0.4.0"):
    raise RuntimeError(
        f"当前qianji-data-mini版本为{installed_qianji}，06号至少需要0.4.0。"
        "请先覆盖本次扩展补丁，运行00_openBB环境构建.ipynb并彻底重启内核。"
    )

try:
    from EmQuantAPI import c as choice_sdk
    print("EmQuantAPI：导入成功")
except Exception as exc:
    raise RuntimeError("当前Notebook内核无法导入EmQuantAPI。") from exc

from qianji_data_mini import Database, ingest_choice_reference

login_mode = os.getenv("CHOICE_LOGIN_MODE", "auto").strip().lower()
username_configured = bool(os.getenv("CHOICE_USERNAME", "").strip())
password_configured = bool(os.getenv("CHOICE_PASSWORD", ""))

print("Choice登录模式：", login_mode)
print("用户名已配置：", username_configured)
print("密码已配置：", password_configured)

if login_mode not in {"auto", "userinfo", "password"}:
    raise RuntimeError("CHOICE_LOGIN_MODE只能是auto、userinfo或password。")
if login_mode == "password" and not (username_configured and password_configured):
    raise RuntimeError("password模式必须同时配置CHOICE_USERNAME和CHOICE_PASSWORD。")

qianji-data-mini版本： 0.4.0
EmQuantAPI：导入成功
Choice登录模式： userinfo
用户名已配置： False
密码已配置： False


## 3. 设置主数据范围和交易日历范围

In [3]:
from datetime import date, timedelta

# sample：三只证券验收（推荐）；all_a：Choice全部A股板块001004。
MASTER_SCOPE = os.getenv("CHOICE_REFERENCE_SCOPE", "sample").strip().lower()
RUN_REAL_CALLS = True
STRICT_MODE = False

raw_symbols = os.getenv(
    "CHOICE_REFERENCE_SYMBOLS",
    "000001.SZ,601988.SH,510300.SH",
)
SAMPLE_SYMBOLS = list(
    dict.fromkeys(item.strip().upper() for item in raw_symbols.split(",") if item.strip())
)
ALL_A_SECTOR_CODE = os.getenv("CHOICE_ALL_A_SECTOR_CODE", "001004").strip()
MARKETS = [
    item.strip().upper()
    for item in os.getenv("CHOICE_CALENDAR_MARKETS", "CNSESH,CNSESZ").split(",")
    if item.strip()
]

default_end = date.today() - timedelta(days=1)
default_start = default_end - timedelta(days=62)
CALENDAR_START_DATE = os.getenv("CHOICE_CALENDAR_START_DATE", "").strip() or default_start.isoformat()
CALENDAR_END_DATE = os.getenv("CHOICE_CALENDAR_END_DATE", "").strip() or default_end.isoformat()
AS_OF_DATE = os.getenv("CHOICE_MASTER_AS_OF_DATE", "").strip() or CALENDAR_END_DATE
BATCH_SIZE = int(os.getenv("CHOICE_MASTER_BATCH_SIZE", "100"))

if MASTER_SCOPE not in {"sample", "all_a"}:
    raise RuntimeError("MASTER_SCOPE只能是sample或all_a。")
if MASTER_SCOPE == "sample" and not SAMPLE_SYMBOLS:
    raise RuntimeError("sample模式至少需要一个证券代码。")
if date.fromisoformat(CALENDAR_START_DATE) > date.fromisoformat(CALENDAR_END_DATE):
    raise RuntimeError("交易日历开始日期不能晚于结束日期。")

database = Database()
DB_PATH = database.path
OUTPUT_DIR = (PROJECT_ROOT / "validation_output").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("真实调用：", RUN_REAL_CALLS)
print("主数据范围：", MASTER_SCOPE)
print("样本证券：", SAMPLE_SYMBOLS if MASTER_SCOPE == "sample" else "全部A股板块" + ALL_A_SECTOR_CODE)
print("交易市场：", MARKETS)
print("交易日历范围：", CALENDAR_START_DATE, "至", CALENDAR_END_DATE)
print("SQLite数据库：", DB_PATH)

真实调用： True
主数据范围： all_a
样本证券： 全部A股板块001004
交易市场： ['CNSESH', 'CNSESZ']
交易日历范围： 2026-06-30 至 2026-08-31
SQLite数据库： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\data\qianji_market.db


## 4. 通用脱敏、计数和数据整理函数

In [4]:
import json
import sqlite3
from typing import Any

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


def safe_error(value: object) -> str:
    text = str(value)
    for secret in [
        os.getenv("CHOICE_USERNAME", ""),
        os.getenv("CHOICE_PASSWORD", ""),
        os.getenv("TUSHARE_TOKEN", ""),
    ]:
        if secret:
            text = text.replace(secret, "***")
    return text[:2000]


def scalar(sql: str, params: list | tuple = ()) -> int:
    with database.connect() as connection:
        return int(connection.execute(sql, params).fetchone()[0])


def master_count() -> int:
    if MASTER_SCOPE == "sample":
        placeholders = ",".join("?" for _ in SAMPLE_SYMBOLS)
        return scalar(
            f"SELECT COUNT(*) FROM security_master WHERE source='choice' AND symbol IN ({placeholders})",
            SAMPLE_SYMBOLS,
        )
    return scalar("SELECT COUNT(*) FROM security_master WHERE source='choice'")


def calendar_count() -> int:
    placeholders = ",".join("?" for _ in MARKETS)
    return scalar(
        f'''
        SELECT COUNT(*) FROM trading_calendar
        WHERE source='choice' AND market IN ({placeholders})
          AND trade_date BETWEEN ? AND ?
        ''',
        [*MARKETS, CALENDAR_START_DATE, CALENDAR_END_DATE],
    )


print("通用函数加载完成。")

通用函数加载完成。


## 5. 检查SQLite表结构和运行前记录数

In [5]:
with database.connect() as connection:
    tables = sorted(
        row[0]
        for row in connection.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        ).fetchall()
    )

required_tables = {"security_master", "trading_calendar", "reference_ingestion_run"}
print("数据库表：", tables)
print("06号所需表完整：", required_tables <= set(tables))
if not required_tables <= set(tables):
    raise RuntimeError("06号所需SQLite表不存在，请先运行00号Notebook重新安装项目。")

master_before = master_count()
calendar_before = calendar_count()
print("运行前主数据记录数：", master_before)
print("运行前日历记录数：", calendar_before)

数据库表： ['daily_bar', 'ingestion_run', 'reference_ingestion_run', 'security_master', 'sqlite_sequence', 'trading_calendar']
06号所需表完整： True
运行前主数据记录数： 3
运行前日历记录数： 126


## 6. Choice真实取数并连续落库两次

In [6]:
first_result = None
second_result = None
first_error = ""
second_error = ""

ingest_kwargs = {
    "symbols": SAMPLE_SYMBOLS if MASTER_SCOPE == "sample" else [],
    "sector_code": None if MASTER_SCOPE == "sample" else ALL_A_SECTOR_CODE,
    "calendar_start_date": CALENDAR_START_DATE,
    "calendar_end_date": CALENDAR_END_DATE,
    "markets": MARKETS,
    "as_of_date": AS_OF_DATE,
    "batch_size": BATCH_SIZE,
    "database_path": DB_PATH,
}

if RUN_REAL_CALLS:
    try:
        first_result = ingest_choice_reference(**ingest_kwargs)
    except Exception as exc:
        first_error = safe_error(f"{type(exc).__name__}: {exc}")

master_after_first = master_count()
calendar_after_first = calendar_count()

if RUN_REAL_CALLS and first_result is not None and not first_result.errors:
    try:
        second_result = ingest_choice_reference(**ingest_kwargs)
    except Exception as exc:
        second_error = safe_error(f"{type(exc).__name__}: {exc}")

master_after_second = master_count()
calendar_after_second = calendar_count()


def result_row(label: str, result, error: str) -> dict:
    if result is None:
        return {
            "run": label,
            "status": "SKIP" if not RUN_REAL_CALLS else "FAIL",
            "security_received": 0,
            "security_stored": 0,
            "calendar_received": 0,
            "calendar_stored": 0,
            "errors": error or "未执行",
        }
    errors = {key: safe_error(value) for key, value in result.errors.items()}
    return {
        "run": label,
        "status": "PASS" if not errors else "PARTIAL",
        "security_received": result.security_received_rows,
        "security_stored": result.security_stored_rows,
        "calendar_received": result.calendar_received_rows,
        "calendar_stored": result.calendar_stored_rows,
        "errors": json.dumps(errors, ensure_ascii=False),
    }


ingestion_runs_df = pd.DataFrame(
    [
        result_row("第一次", first_result, first_error),
        result_row("第二次", second_result, second_error),
    ]
)
idempotency_df = pd.DataFrame(
    [
        {
            "dataset": "security_master",
            "before": master_before,
            "after_first": master_after_first,
            "after_second": master_after_second,
            "second_minus_first": master_after_second - master_after_first,
            "idempotent": master_after_first > 0 and master_after_first == master_after_second,
        },
        {
            "dataset": "trading_calendar",
            "before": calendar_before,
            "after_first": calendar_after_first,
            "after_second": calendar_after_second,
            "second_minus_first": calendar_after_second - calendar_after_first,
            "idempotent": calendar_after_first > 0 and calendar_after_first == calendar_after_second,
        },
    ]
)

display(ingestion_runs_df)
display(idempotency_df)

[EmQuantAPI Python] [Em_Info][2026-09-01 00:30:52]:The current version is EmQuantAPI(V2.7.5.0).

[EmQuantAPI Python] [Em_Info][2026-09-01 00:30:52]:verifying your token...

[EmQuantAPI Python] [Em_Info][2026-09-01 00:30:52]:connect server...

[EmQuantAPI Python] [Em_Info][2026-09-01 00:30:56]:token login start success!

[EmQuantAPI Python] [Em_Info][2026-09-01 00:31:00]:updating ChoiceToHQ.xml from version 0 to 120

[EmQuantAPI Python] [Em_Info][2026-09-01 00:31:02]:loading ChoiceToHQ.xml...

[EmQuantAPI Python] [Em_Info][2026-09-01 00:31:10]:DownLoad D:/EMQuantAPI_Python/python3/libs/windows/bjse_code_conversion.txt success.

[EmQuantAPI Python] [Em_Info][2026-09-01 00:31:14]:percentflag(for csd/css/cses) update success.

[EmQuantAPI Python] [Em_Info][2026-09-01 00:33:20]:heartbeatthread end.

[EmQuantAPI Python] [Em_Info][2026-09-01 00:33:22]:The current version is EmQuantAPI(V2.7.5.0).

[EmQuantAPI Python] [Em_Info][2026-09-01 00:33:22]:verifying your token...

[EmQuantAPI Python] [

,run,status,security_received,security_stored,calendar_received,calendar_stored,errors
0,第一次,PASS,5212,5212,126,126,{}
1,第二次,PASS,5212,5212,126,126,{}


,dataset,before,after_first,after_second,second_minus_first,idempotent
0,security_master,3,5213,5213,0,True
1,trading_calendar,126,126,126,0,True


## 7. 读取并验收证券主数据

In [7]:
validated_symbols = (
    first_result.requested_symbols
    if first_result is not None and first_result.requested_symbols
    else SAMPLE_SYMBOLS
)
master_df = database.query_security_master(
    source="choice",
    symbols=validated_symbols if MASTER_SCOPE == "sample" else None,
)

required_master_fields = [
    "source", "symbol", "name", "exchange", "asset_type",
    "currency", "status", "as_of_date", "fetched_at",
]
master_duplicates = int(master_df.duplicated(subset=["source", "symbol"]).sum()) if not master_df.empty else None
master_missing_required = int(master_df[required_master_fields].isna().any(axis=1).sum()) if not master_df.empty else None
name_fallback = int((master_df["name"].astype(str) == master_df["symbol"].astype(str)).sum()) if not master_df.empty else None
invalid_symbols = int((~master_df["symbol"].astype(str).str.match(r"^[0-9A-Z]+\.(SH|SZ|BJ)$")).sum()) if not master_df.empty else None

expected_exchange = master_df["symbol"].str.rsplit(".", n=1).str[-1].map(
    {"SH": "SSE", "SZ": "SZSE", "BJ": "BSE"}
) if not master_df.empty else pd.Series(dtype="object")
exchange_mismatch = int((expected_exchange != master_df["exchange"]).sum()) if not master_df.empty else None
list_date_missing = int(master_df["list_date"].isna().sum()) if not master_df.empty else None
delist_date_missing = int(master_df["delist_date"].isna().sum()) if not master_df.empty else None

master_quality_df = pd.DataFrame(
    [
        ["主数据非空", ">0", len(master_df), len(master_df) > 0, True],
        ["主键重复", "=0", master_duplicates, master_duplicates == 0, True],
        ["关键字段缺失", "=0", master_missing_required, master_missing_required == 0, True],
        ["名称使用代码回退", "=0", name_fallback, name_fallback == 0, True],
        ["证券代码格式异常", "=0", invalid_symbols, invalid_symbols == 0, True],
        ["交易所映射不一致", "=0", exchange_mismatch, exchange_mismatch == 0, True],
        ["上市日期缺失", "记录即可", list_date_missing, True, False],
        ["退市日期缺失", "正常上市证券允许为空", delist_date_missing, True, False],
    ],
    columns=["check", "threshold", "actual", "passed", "hard_gate"],
)

display(master_quality_df)
display(master_df.head(20))

,check,threshold,actual,passed,hard_gate
0,主数据非空,>0,5213,True,True
1,主键重复,=0,0,True,True
2,关键字段缺失,=0,0,True,True
3,名称使用代码回退,=0,0,True,True
4,证券代码格式异常,=0,0,True,True
5,交易所映射不一致,=0,0,True,True
6,上市日期缺失,记录即可,1,True,False
7,退市日期缺失,正常上市证券允许为空,5213,True,False


,source,symbol,name,exchange,asset_type,currency,list_date,delist_date,status,as_of_date,fetched_at
0,choice,510300.SH,华泰柏瑞沪深300ETF,SSE,etf,CNY,None,None,active,2026-08-31,2026-09-01T07:17:15.195244+00:00
1,choice,600000.SH,浦发银行,SSE,equity,CNY,1999-11-10,None,active,2026-08-31,2026-09-01T07:34:08.940656+00:00
2,choice,600004.SH,白云机场,SSE,equity,CNY,2003-04-28,None,active,2026-08-31,2026-09-01T07:34:08.940656+00:00
3,choice,600006.SH,东风股份,SSE,equity,CNY,1999-07-27,None,active,2026-08-31,2026-09-01T07:34:08.940656+00:00
4,choice,600007.SH,中国国贸,SSE,equity,CNY,1999-03-12,None,active,2026-08-31,2026-09-01T07:34:10.003513+00:00
5,choice,600008.SH,首创环保,SSE,equity,CNY,2000-04-27,None,active,2026-08-31,2026-09-01T07:34:10.003513+00:00
6,choice,600009.SH,上海机场,SSE,equity,CNY,1998-02-18,None,active,2026-08-31,2026-09-01T07:34:10.003513+00:00
7,choice,600010.SH,包钢股份,SSE,equity,CNY,2001-03-09,None,active,2026-08-31,2026-09-01T07:34:10.003513+00:00
8,choice,600011.SH,华能国际,SSE,equity,CNY,2001-12-06,None,active,2026-08-31,2026-09-01T07:34:10.003513+00:00
9,choice,600012.SH,皖通高速,SSE,equity,CNY,2003-01-07,None,active,2026-08-31,2026-09-01T07:34:10.003513+00:00


## 8. 读取并验收沪深交易日历

In [8]:
calendar_frames = []
for market in MARKETS:
    frame = database.query_trading_calendar(
        source="choice",
        market=market,
        start_date=CALENDAR_START_DATE,
        end_date=CALENDAR_END_DATE,
    )
    calendar_frames.append(frame)

calendar_df = pd.concat(calendar_frames, ignore_index=True) if calendar_frames else pd.DataFrame()
if not calendar_df.empty:
    calendar_df["trade_date"] = pd.to_datetime(calendar_df["trade_date"], errors="coerce")
    calendar_df["weekday"] = calendar_df["trade_date"].dt.dayofweek

expected_days_per_market = (
    date.fromisoformat(CALENDAR_END_DATE) - date.fromisoformat(CALENDAR_START_DATE)
).days + 1
expected_calendar_rows = expected_days_per_market * len(MARKETS)
calendar_duplicates = int(calendar_df.duplicated(subset=["source", "market", "trade_date"]).sum()) if not calendar_df.empty else None
calendar_bad_dates = int(calendar_df["trade_date"].isna().sum()) if not calendar_df.empty else None
calendar_bad_open_flag = int((~calendar_df["is_open"].isin([0, 1])).sum()) if not calendar_df.empty else None
weekend_open = int(((calendar_df["weekday"] >= 5) & (calendar_df["is_open"] == 1)).sum()) if not calendar_df.empty else None
open_days = int(calendar_df["is_open"].sum()) if not calendar_df.empty else 0

market_counts = calendar_df.groupby("market").size().to_dict() if not calendar_df.empty else {}
market_coverage_ok = all(market_counts.get(market, 0) == expected_days_per_market for market in MARKETS)

shsz_open_difference = None
if {"CNSESH", "CNSESZ"} <= set(MARKETS) and not calendar_df.empty:
    sh_dates = set(calendar_df.loc[(calendar_df["market"] == "CNSESH") & (calendar_df["is_open"] == 1), "trade_date"])
    sz_dates = set(calendar_df.loc[(calendar_df["market"] == "CNSESZ") & (calendar_df["is_open"] == 1), "trade_date"])
    shsz_open_difference = len(sh_dates.symmetric_difference(sz_dates))

calendar_quality_df = pd.DataFrame(
    [
        ["完整自然日记录数", f"={expected_calendar_rows}", len(calendar_df), len(calendar_df) == expected_calendar_rows, True],
        ["覆盖全部市场", f"每市场{expected_days_per_market}日", json.dumps(market_counts, ensure_ascii=False), market_coverage_ok, True],
        ["主键重复", "=0", calendar_duplicates, calendar_duplicates == 0, True],
        ["日期解析失败", "=0", calendar_bad_dates, calendar_bad_dates == 0, True],
        ["开市标记异常", "=0", calendar_bad_open_flag, calendar_bad_open_flag == 0, True],
        ["交易日非空", ">0", open_days, open_days > 0, True],
        ["周末被标为开市", "=0", weekend_open, weekend_open == 0, True],
        ["沪深开市日差异", "通常=0", shsz_open_difference if shsz_open_difference is not None else "不适用", shsz_open_difference in {None, 0}, False],
    ],
    columns=["check", "threshold", "actual", "passed", "hard_gate"],
)

display(calendar_quality_df)
display(calendar_df.head(20))

,check,threshold,actual,passed,hard_gate
0,完整自然日记录数,=126,126,True,True
1,覆盖全部市场,每市场63日,"{""CNSESH"": 63, ""CNSESZ"": 63}",True,True
2,主键重复,=0,0,True,True
3,日期解析失败,=0,0,True,True
4,开市标记异常,=0,0,True,True
5,交易日非空,>0,90,True,True
6,周末被标为开市,=0,0,True,True
7,沪深开市日差异,通常=0,0,True,False


,source,market,trade_date,is_open,previous_open_date,next_open_date,timezone,fetched_at,weekday
0,choice,CNSESH,2026-06-30,1,None,2026-07-01,Asia/Shanghai,2026-09-01T07:34:45.061124+00:00,1
1,choice,CNSESH,2026-07-01,1,2026-06-30,2026-07-02,Asia/Shanghai,2026-09-01T07:34:45.061124+00:00,2
2,choice,CNSESH,2026-07-02,1,2026-07-01,2026-07-03,Asia/Shanghai,2026-09-01T07:34:45.061124+00:00,3
3,choice,CNSESH,2026-07-03,1,2026-07-02,2026-07-06,Asia/Shanghai,2026-09-01T07:34:45.061124+00:00,4
4,choice,CNSESH,2026-07-04,0,2026-07-03,2026-07-06,Asia/Shanghai,2026-09-01T07:34:45.061124+00:00,5
5,choice,CNSESH,2026-07-05,0,2026-07-03,2026-07-06,Asia/Shanghai,2026-09-01T07:34:45.061124+00:00,6
6,choice,CNSESH,2026-07-06,1,2026-07-03,2026-07-07,Asia/Shanghai,2026-09-01T07:34:45.061124+00:00,0
7,choice,CNSESH,2026-07-07,1,2026-07-06,2026-07-08,Asia/Shanghai,2026-09-01T07:34:45.061124+00:00,1
8,choice,CNSESH,2026-07-08,1,2026-07-07,2026-07-09,Asia/Shanghai,2026-09-01T07:34:45.061124+00:00,2
9,choice,CNSESH,2026-07-09,1,2026-07-08,2026-07-10,Asia/Shanghai,2026-09-01T07:34:45.061124+00:00,3


## 9. 使用已落库Choice日线反查交易日历

In [9]:
with database.connect() as connection:
    daily_dates_df = pd.read_sql_query(
        '''
        SELECT DISTINCT symbol, trade_date
        FROM daily_bar
        WHERE source='choice' AND trade_date BETWEEN ? AND ?
        ORDER BY symbol, trade_date
        ''',
        connection,
        params=[CALENDAR_START_DATE, CALENDAR_END_DATE],
    )

if not daily_dates_df.empty:
    daily_dates_df["market"] = np.where(
        daily_dates_df["symbol"].str.endswith(".SH"),
        "CNSESH",
        np.where(daily_dates_df["symbol"].str.endswith(".SZ"), "CNSESZ", ""),
    )
    daily_dates_df["trade_date"] = pd.to_datetime(daily_dates_df["trade_date"], errors="coerce")
    calendar_join = calendar_df[["market", "trade_date", "is_open"]].copy()
    daily_calendar_check_df = daily_dates_df.merge(
        calendar_join,
        on=["market", "trade_date"],
        how="left",
    )
    daily_calendar_check_df["check"] = np.where(
        daily_calendar_check_df["is_open"] == 1,
        "PASS",
        "FAIL",
    )
else:
    daily_calendar_check_df = pd.DataFrame(
        columns=["symbol", "trade_date", "market", "is_open", "check"]
    )

daily_calendar_failures = int((daily_calendar_check_df["check"] == "FAIL").sum()) if not daily_calendar_check_df.empty else None
print("关联到的Choice日线日期数：", len(daily_calendar_check_df))
print("日线落在非交易日或未匹配日历的数量：", daily_calendar_failures)
display(daily_calendar_check_df.head(20))

关联到的Choice日线日期数： 96
日线落在非交易日或未匹配日历的数量： 0


,symbol,trade_date,market,is_open,check
0,000001.SZ,2026-07-16,CNSESZ,1,PASS
1,000001.SZ,2026-07-17,CNSESZ,1,PASS
2,000001.SZ,2026-07-20,CNSESZ,1,PASS
3,000001.SZ,2026-07-21,CNSESZ,1,PASS
4,000001.SZ,2026-07-22,CNSESZ,1,PASS
5,000001.SZ,2026-07-23,CNSESZ,1,PASS
6,000001.SZ,2026-07-24,CNSESZ,1,PASS
7,000001.SZ,2026-07-27,CNSESZ,1,PASS
8,000001.SZ,2026-07-28,CNSESZ,1,PASS
9,000001.SZ,2026-07-29,CNSESZ,1,PASS


## 10. 汇总硬性验收门槛

In [10]:
quality_rows = []


def add_gate(category: str, check: str, threshold: str, actual: object, passed: bool, evidence: str):
    quality_rows.append(
        {
            "category": category,
            "check": check,
            "threshold": threshold,
            "actual": actual,
            "status": "PASS" if passed else "FAIL",
            "evidence": evidence,
        }
    )


first_errors = first_result.errors if first_result is not None else {"run": first_error or "未执行"}
second_errors = second_result.errors if second_result is not None else {"run": second_error or "未执行"}
add_gate("真实调用", "第一次取数无错误", "错误数=0", len(first_errors), len(first_errors) == 0, "ingestion_runs")
add_gate("真实调用", "第二次取数无错误", "错误数=0", len(second_errors), len(second_errors) == 0, "ingestion_runs")

if MASTER_SCOPE == "sample":
    master_coverage = set(SAMPLE_SYMBOLS) <= set(master_df["symbol"]) if not master_df.empty else False
    add_gate("证券主数据", "覆盖全部样本证券", f"={len(SAMPLE_SYMBOLS)}只", len(master_df), master_coverage, "security_master")
else:
    add_gate("证券主数据", "全部A股板块返回非空", ">0", len(master_df), len(master_df) > 0, "security_master")

for row in master_quality_df.loc[master_quality_df["hard_gate"]].to_dict(orient="records"):
    add_gate("证券主数据", row["check"], str(row["threshold"]), row["actual"], bool(row["passed"]), "master_quality")

for row in calendar_quality_df.loc[calendar_quality_df["hard_gate"]].to_dict(orient="records"):
    add_gate("交易日历", row["check"], str(row["threshold"]), row["actual"], bool(row["passed"]), "calendar_quality")

for row in idempotency_df.to_dict(orient="records"):
    add_gate("幂等落库", f"{row['dataset']}重复运行不增行", "second_minus_first=0", row["second_minus_first"], bool(row["idempotent"]), "idempotency")

with database.connect() as connection:
    sqlite_integrity = str(connection.execute("PRAGMA quick_check").fetchone()[0])
add_gate("SQLite", "数据库完整性", "ok", sqlite_integrity, sqlite_integrity == "ok", "database")

if not daily_calendar_check_df.empty:
    add_gate("关联校验", "Choice日线均落在开市日", "失败数=0", daily_calendar_failures, daily_calendar_failures == 0, "daily_calendar_check")

quality_gates_df = pd.DataFrame(quality_rows)
passed_gates = int((quality_gates_df["status"] == "PASS").sum())
failed_gates = int((quality_gates_df["status"] == "FAIL").sum())

print("PASS数量：", passed_gates)
print("FAIL数量：", failed_gates)
display(quality_gates_df)

PASS数量： 20
FAIL数量： 0


,category,check,threshold,actual,status,evidence
0,真实调用,第一次取数无错误,错误数=0,0,PASS,ingestion_runs
1,真实调用,第二次取数无错误,错误数=0,0,PASS,ingestion_runs
2,证券主数据,全部A股板块返回非空,>0,5213,PASS,security_master
3,证券主数据,主数据非空,>0,5213,PASS,master_quality
4,证券主数据,主键重复,=0,0,PASS,master_quality
5,证券主数据,关键字段缺失,=0,0,PASS,master_quality
6,证券主数据,名称使用代码回退,=0,0,PASS,master_quality
7,证券主数据,证券代码格式异常,=0,0,PASS,master_quality
8,证券主数据,交易所映射不一致,=0,0,PASS,master_quality
9,交易日历,完整自然日记录数,=126,126,PASS,calendar_quality


## 11. 更新Choice数据地图

In [11]:
data_map_df = pd.DataFrame(
    [
        {
            "dataset": "security_master",
            "source_function": "Choice css；全量代码可选sector(001004)",
            "storage_table": "security_master",
            "primary_key": "source + symbol",
            "coverage": f"{len(master_df)}只证券",
            "update_cadence": "每日或上市状态变化后",
            "required_fields": "symbol,name,exchange,asset_type,currency,status,as_of_date",
            "optional_fields": "list_date,delist_date",
            "python_access": "Database.query_security_master",
            "openbb_mapping": "后续映射EquitySearch；本期先完成公司库落地",
            "status": "PASS" if not master_df.empty and master_missing_required == 0 else "FAIL",
        },
        {
            "dataset": "trading_calendar",
            "source_function": "Choice tradedates",
            "storage_table": "trading_calendar",
            "primary_key": "source + market + trade_date",
            "coverage": f"{len(MARKETS)}市场，{len(calendar_df)}个自然日",
            "update_cadence": "每年初始化并按官方调整更新",
            "required_fields": "market,trade_date,is_open,timezone",
            "optional_fields": "previous_open_date,next_open_date",
            "python_access": "Database.query_trading_calendar",
            "openbb_mapping": "暂无稳定标准模型；先作为qianji内部参考数据接口",
            "status": "PASS" if len(calendar_df) == expected_calendar_rows else "FAIL",
        },
    ]
)
display(data_map_df)

,dataset,source_function,storage_table,primary_key,coverage,update_cadence,required_fields,optional_fields,python_access,openbb_mapping,status
0,security_master,Choice css；全量代码可选sector(001004),security_master,source + symbol,5213只证券,每日或上市状态变化后,"symbol,name,exchange,asset_type,currency,statu...","list_date,delist_date",Database.query_security_master,后续映射EquitySearch；本期先完成公司库落地,PASS
1,trading_calendar,Choice tradedates,trading_calendar,source + market + trade_date,2市场，126个自然日,每年初始化并按官方调整更新,"market,trade_date,is_open,timezone","previous_open_date,next_open_date",Database.query_trading_calendar,暂无稳定标准模型；先作为qianji内部参考数据接口,PASS


## 12. 导出Excel和JSON验收证据

In [12]:
from datetime import datetime

from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


timestamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
excel_path = OUTPUT_DIR / f"Choice证券主数据交易日历验收_{timestamp}.xlsx"
json_path = OUTPUT_DIR / f"Choice证券主数据交易日历验收_{timestamp}.json"

overview_df = pd.DataFrame(
    [
        ["generated_at", pd.Timestamp.now(tz="UTC").isoformat()],
        ["python", sys.executable],
        ["project_root", str(PROJECT_ROOT)],
        ["database", str(DB_PATH)],
        ["qianji_data_mini_version", package_version("qianji-data-mini")],
        ["master_scope", MASTER_SCOPE],
        ["security_rows", len(master_df)],
        ["calendar_markets", ",".join(MARKETS)],
        ["calendar_range", f"{CALENDAR_START_DATE} 至 {CALENDAR_END_DATE}"],
        ["calendar_rows", len(calendar_df)],
        ["passed_gates", passed_gates],
        ["failed_gates", failed_gates],
        ["credentials_included", False],
        ["official_document", "https://quantapi.eastmoney.com/Upload/EMQuantAPI_Python.html"],
        ["boundary", "本期完成公司库落地；OpenBB EquitySearch映射留作下一步"],
    ],
    columns=["item", "value"],
)

reference_status_df = database.reference_status()
sheet_frames = {
    "验收概览": overview_df,
    "质量门槛": quality_gates_df,
    "落库执行": ingestion_runs_df,
    "幂等检查": idempotency_df,
    "证券主数据": master_df,
    "主数据质量": master_quality_df,
    "交易日历": calendar_df,
    "日历质量": calendar_quality_df,
    "日线关联": daily_calendar_check_df,
    "数据地图": data_map_df,
    "参考数据状态": reference_status_df,
}


def safe_cell(value):
    if isinstance(value, str) and value[:1] in {"=", "+", "-", "@"}:
        return "'" + value
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    return value


safe_frames = {}
for sheet_name, frame in sheet_frames.items():
    safe_frame = frame.copy()
    for column in safe_frame.columns:
        safe_frame[column] = safe_frame[column].map(safe_cell)
    safe_frames[sheet_name] = safe_frame

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, frame in safe_frames.items():
        frame.to_excel(writer, sheet_name=sheet_name[:31], index=False, startrow=3)

workbook = load_workbook(excel_path)
NAVY, BLUE, LIGHT_BLUE = "17365D", "2F75B5", "D9EAF7"
GREEN, YELLOW, RED, WHITE, GRID = "E2F0D9", "FFF2CC", "FCE4D6", "FFFFFF", "B7C9D6"
thin = Side(style="thin", color=GRID)

for worksheet in workbook.worksheets:
    frame = safe_frames[worksheet.title]
    max_col = max(1, len(frame.columns))
    max_row = worksheet.max_row
    last_col = get_column_letter(max_col)

    worksheet.merge_cells(start_row=1, start_column=1, end_row=1, end_column=max_col)
    title = worksheet.cell(1, 1, f"Choice证券主数据与交易日历验收｜{worksheet.title}")
    title.fill = PatternFill("solid", fgColor=NAVY)
    title.font = Font(name="Microsoft YaHei", size=15, bold=True, color=WHITE)
    title.alignment = Alignment(vertical="center")
    worksheet.row_dimensions[1].height = 28

    worksheet.merge_cells(start_row=2, start_column=1, end_row=2, end_column=max_col)
    subtitle = worksheet.cell(2, 1, "真实结果与自动验收证据；凭据未导出")
    subtitle.fill = PatternFill("solid", fgColor=LIGHT_BLUE)
    subtitle.font = Font(name="Microsoft YaHei", size=10, color=NAVY)

    for cell in worksheet[4]:
        cell.fill = PatternFill("solid", fgColor=BLUE)
        cell.font = Font(name="Microsoft YaHei", bold=True, color=WHITE)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = Border(top=thin, bottom=thin, left=thin, right=thin)

    for row in worksheet.iter_rows(min_row=5, max_row=max_row, max_col=max_col):
        for cell in row:
            cell.font = Font(name="Microsoft YaHei", size=10)
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = Border(top=thin, bottom=thin, left=thin, right=thin)
            text = str(cell.value or "")
            if text in {"PASS", "True"}:
                cell.fill = PatternFill("solid", fgColor=GREEN)
            elif text.startswith("FAIL") or text == "False":
                cell.fill = PatternFill("solid", fgColor=RED)
            elif text in {"SKIP", "PARTIAL"}:
                cell.fill = PatternFill("solid", fgColor=YELLOW)

    for column_index, column_name in enumerate(frame.columns, start=1):
        values = [str(column_name)] + [str(value or "") for value in frame[column_name].head(200)]
        longest = max((len(value) for value in values), default=8)
        worksheet.column_dimensions[get_column_letter(column_index)].width = min(max(longest * 1.1 + 2, 10), 42)

    worksheet.freeze_panes = "A5"
    worksheet.auto_filter.ref = f"A4:{last_col}{max_row}"
    worksheet.sheet_view.showGridLines = False
    worksheet.print_title_rows = "1:4"
    worksheet.page_setup.orientation = "landscape"
    worksheet.page_setup.fitToWidth = 1
    worksheet.sheet_properties.pageSetUpPr.fitToPage = True

workbook.save(excel_path)


def records(frame: pd.DataFrame) -> list[dict]:
    safe = frame.copy()
    for column in safe.columns:
        safe[column] = safe[column].map(
            lambda value: value.isoformat() if hasattr(value, "isoformat") else value
        )
    return safe.where(pd.notna(safe), None).to_dict(orient="records")


json_payload = {
    "generated_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "python": sys.executable,
    "project_root": str(PROJECT_ROOT),
    "database": str(DB_PATH),
    "qianji_data_mini_version": package_version("qianji-data-mini"),
    "master_scope": MASTER_SCOPE,
    "requested_symbols": validated_symbols,
    "markets": MARKETS,
    "calendar_range": [CALENDAR_START_DATE, CALENDAR_END_DATE],
    "credentials_included": False,
    "ingestion_runs": records(ingestion_runs_df),
    "idempotency": records(idempotency_df),
    "security_master": records(master_df),
    "master_quality": records(master_quality_df),
    "trading_calendar": records(calendar_df),
    "calendar_quality": records(calendar_quality_df),
    "daily_calendar_check": records(daily_calendar_check_df),
    "data_map": records(data_map_df),
    "quality_gates": records(quality_gates_df),
    "passed_gates": passed_gates,
    "failed_gates": failed_gates,
}
json_path.write_text(
    json.dumps(json_payload, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print("Excel已生成：", excel_path)
print("JSON已生成：", json_path)
print("Excel大小：", excel_path.stat().st_size)
print("JSON大小：", json_path.stat().st_size)

Excel已生成： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice证券主数据交易日历验收_20260901_003448.xlsx
JSON已生成： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice证券主数据交易日历验收_20260901_003448.json
Excel大小： 311371
JSON大小： 2068207


## 13. 自动复核输出并给出结论

In [13]:
check_workbook = load_workbook(excel_path, read_only=True, data_only=False)
expected_sheets = list(safe_frames)
actual_sheets = check_workbook.sheetnames
missing_sheets = sorted(set(expected_sheets) - set(actual_sheets))
json_check = json.loads(json_path.read_text(encoding="utf-8"))

export_checks_df = pd.DataFrame(
    [
        ["Excel文件存在", excel_path.exists(), str(excel_path)],
        ["JSON文件存在", json_path.exists(), str(json_path)],
        ["工作表完整", not missing_sheets, f"缺少：{missing_sheets}" if missing_sheets else f"共{len(actual_sheets)}张表"],
        ["凭据未导出", json_check.get("credentials_included") is False, "credentials_included=False"],
        ["质量门槛已生成", len(quality_gates_df) > 0, f"PASS={passed_gates}, FAIL={failed_gates}"],
    ],
    columns=["check", "passed", "detail"],
)
display(export_checks_df)

if not export_checks_df["passed"].all():
    raise RuntimeError("导出文件复核失败，请查看上表。")

failed_checks = quality_gates_df.loc[
    quality_gates_df["status"] == "FAIL",
    ["category", "check", "actual"],
]
if failed_checks.empty:
    print("最终结论：证券主数据与交易日历全部硬性验收指标通过。")
else:
    print("最终结论：证据已导出，但仍有未通过项目：")
    display(failed_checks)
    if STRICT_MODE:
        raise RuntimeError(f"存在{len(failed_checks)}项硬性验收失败。")

,check,passed,detail
0,Excel文件存在,True,D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\valid...
1,JSON文件存在,True,D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\valid...
2,工作表完整,True,共11张表
3,凭据未导出,True,credentials_included=False
4,质量门槛已生成,True,"PASS=20, FAIL=0"


最终结论：证券主数据与交易日历全部硬性验收指标通过。
